In [1]:
pip install polars

   ---------------------------------------- 0.0/828.7 kB ? eta -:--:--
   ---------------------------------------- 828.7/828.7 kB 8.6 MB/s  0:00:00
   ---------------------------------------- 0.0/51.8 MB ? eta -:--:--
   -- ------------------------------------- 3.4/51.8 MB 28.2 MB/s eta 0:00:02
   --------- ------------------------------ 12.6/51.8 MB 29.5 MB/s eta 0:00:02
   -------------- ------------------------- 18.9/51.8 MB 29.4 MB/s eta 0:00:02
   ------------------ --------------------- 23.9/51.8 MB 28.1 MB/s eta 0:00:01
   --------------------- ------------------ 27.8/51.8 MB 26.5 MB/s eta 0:00:01
   -------------------------- ------------- 34.6/51.8 MB 27.1 MB/s eta 0:00:01
   ------------------------------- -------- 41.2/51.8 MB 27.7 MB/s eta 0:00:01
   ----------------------------------- ---- 46.4/51.8 MB 27.3 MB/s eta 0:00:01
   ---------------------------------------  51.6/51.8 MB 27.5 MB/s eta 0:00:01
   ---------------------------------------- 51.8/51.8 MB 25.6 MB/s  0:00


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import polars as pl
import joblib
import numpy as np
import os
import zipfile
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
import pandas as pd
from tqdm import tqdm

In [2]:
zip_path = "../data/preprocessed_data.zip" 
extract_to = "../data"                     
csv_file_name = "preprocessed_data.csv"      
csv_path = os.path.join(extract_to, csv_file_name)
if not os.path.exists(csv_path):
    print(f"--- Đang giải nén file {zip_path} ---")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Giải nén xong!")
else:
    print("File CSV đã tồn tại, không cần giải nén lại.")

df = pl.scan_csv(csv_path)

--- Đang giải nén file ../data/preprocessed_data.zip ---
Giải nén xong!


In [ ]:
## Tìm window để chuyển sequence

In [3]:
# Đếm số giao dịch mỗi account theo combined view
sender_counts = (
    df.group_by("Sender_account")
    .agg(pl.len().alias("count"))
    .rename({"Sender_account": "account"})
)

receiver_counts = (
    df.group_by("Receiver_account")
    .agg(pl.len().alias("count"))
    .rename({"Receiver_account": "account"})
)

combined_counts = (
    pl.concat([sender_counts, receiver_counts])
    .group_by("account")
    .agg(pl.sum("count").alias("total_tx"))
    .collect()
)

print(combined_counts["total_tx"].describe())
print("Accounts >= 5 tx :", (combined_counts["total_tx"] >= 5).sum())
print("Accounts >= 10 tx:", (combined_counts["total_tx"] >= 10).sum())
print("Accounts >= 20 tx:", (combined_counts["total_tx"] >= 20).sum())

shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 401853.0  │
│ null_count ┆ 0.0       │
│ mean       ┆ 7.489565  │
│ std        ┆ 24.019372 │
│ min        ┆ 1.0       │
│ 25%        ┆ 1.0       │
│ 50%        ┆ 2.0       │
│ 75%        ┆ 5.0       │
│ max        ┆ 521.0     │
└────────────┴───────────┘
Accounts >= 5 tx : 103064
Accounts >= 10 tx: 78035
Accounts >= 20 tx: 19257


* Window = 20: Chỉ có 19.257 account (chiếm khoảng $4.8\%$) đủ điều kiện. Nếu chọn số này, bạn sẽ vứt bỏ hơn $95\%$ dữ liệu của mình. Rất lãng phí!
* Window = 5: Có 103.064 account. Dữ liệu nhiều nhưng chuỗi 5 giao dịch thì hơi ngắn, LSTM có thể chưa kịp "học" được các pattern phức tạp của tội phạm rửa tiền (thường qua nhiều bước).
* Window = 10: Có 78.035 account. Đây là "điểm ngọt" (Sweet Spot). Bạn giữ được một lượng account đủ lớn (gần 80k) và chuỗi 10 giao dịch là đủ dài để mô hình thấy được sự thay đổi hành vi.

=> Chọn Window = 10

Số lượng sequence: Với 78k account, sau khi dùng sliding window,  sẽ có tổng cộng vài trăm ngàn đến cả triệu sequence (vì mỗi account có $\ge 10$ giao dịch sẽ tạo ra ít nhất 1 sequence, account có 521 giao dịch sẽ tạo ra tới 512 sequences).Độ sâu hành vi: 10 bước đủ để bắt được các vòng lặp (Cycle) hoặc gom tiền (Fan-in) cơ bản.

## Feature Engineering

In [ ]:
WINDOW_SIZE = 10
FEATURE_COLS = [
    'amount_log', 'hour', 'day_of_week', 'currency_match',
    'Payment_type_enc', 'Sender_bank_location_enc', 
    'Receiver_bank_location_enc', 'Payment_currency_enc',
    'Received_currency_enc', 'role'
]

# ---  FEATURE ENGINEERING ---
df = df.with_columns([
    (pl.col("Date").cast(pl.Utf8) + " " + pl.col("Time").cast(pl.Utf8))
    .str.to_datetime("%Y-%m-%d %H:%M:%S%.9f", strict=False)
    .alias("timestamp"),
    pl.col("Amount").log1p().alias("amount_log"),
    (pl.col("Payment_currency") == pl.col("Received_currency"))
    .cast(pl.Int8).alias("currency_match")
])

df = df.with_columns([
    pl.col("timestamp").dt.hour().alias("hour"),
    pl.col("timestamp").dt.weekday().alias("day_of_week")
])

pdf = df.collect().to_pandas()

cat_cols = [
    "Payment_type", "Sender_bank_location", "Receiver_bank_location",
    "Payment_currency", "Received_currency"
]
le = LabelEncoder()
for col in cat_cols:
    pdf[f'{col}_enc'] = le.fit_transform(pdf[col].astype(str))

print("Feature Engineering xong!")



--- Đang xử lý Feature Engineering ---
Feature Engineering xong!


## Chuyển Sequence

In [ ]:
print("--- Đang tạo Combined View ---")
sender_view = pdf.copy()
sender_view['account'] = pdf['Sender_account']
sender_view['role'] = 0

receiver_view = pdf.copy()
receiver_view['account'] = pdf['Receiver_account']
receiver_view['role'] = 1

combined = pd.concat([sender_view, receiver_view], ignore_index=True)
combined = combined.sort_values(['account', 'timestamp']).reset_index(drop=True)
del sender_view, receiver_view

# SCALING 
print("--- Đang Scale dữ liệu về khoảng [0, 1] ---")
scaler = MinMaxScaler()
combined[FEATURE_COLS] = scaler.fit_transform(combined[FEATURE_COLS].astype('float32'))

print(f" Tổng số dòng sau khi gộp: {len(combined)}")

--- Đang tạo Combined View ---
--- Đang Scale dữ liệu về khoảng [0, 1] ---
 Tổng số dòng sau khi gộp: 3009704


In [ ]:
# SLIDING WINDOW ---
print(f"--- Đang cắt Sequence (Window={WINDOW_SIZE}) ---")
sequences, labels = [], []

for account_id, group in tqdm(combined.groupby('account')):
    if len(group) < WINDOW_SIZE:
        continue
    
    feats = group[FEATURE_COLS].values
    target = group['Is_laundering'].values
    
    for i in range(len(feats) - WINDOW_SIZE + 1):
        sequences.append(feats[i : i + WINDOW_SIZE])
        labels.append(1 if target[i : i + WINDOW_SIZE].any() else 0)

X = np.array(sequences, dtype='float32')
y = np.array(labels, dtype='int8')

print(f" X shape: {X.shape}")
print(f" y shape: {y.shape}")
print(f" Tỷ lệ laundering trong sequence: {np.mean(y)*100:.4f}%")


--- Đang cắt Sequence (Window=10) ---


100%|██████████| 401853/401853 [01:18<00:00, 5138.16it/s]


 X shape: (1568448, 10, 10)
 y shape: (1568448,)
 Tỷ lệ laundering trong sequence: 0.4229%


In [ ]:
output_dir = "../data"
file_name = "sequenced_data.npz"
output_path = os.path.join(output_dir, file_name)

np.savez_compressed(output_path, X=X, y=y)

print(f"\n Đã lưu xong file tại: {output_path}")


 Đã lưu xong file tại: ../data\sequences_win10_final.npz


In [ ]:
## Lưu đê dùng cho streaming
joblib.dump(scaler, "../data/scaler.pkl")
joblib.dump(le, "../data/label_encoder.pkl") 
print("Đã lưu scaler!")

In [ ]:
# TÍNH CLASS WEIGHT
neg = len(y) - sum(y)
pos = sum(y)
weight_for_0 = (1 / neg) * (len(y) / 2.0)
weight_for_1 = (1 / pos) * (len(y) / 2.0)
print(f"\nclass_weight:\n{{0: {weight_for_0:.2f}, 1: {weight_for_1:.2f}}}")